# Finance Guidance Assistant — RAG Prototype (Colab)

Run all cells top to bottom. Make sure a GPU runtime is selected:
**Runtime → Change runtime type → Hardware accelerator → GPU (T4)**

This notebook:
1. Mounts Google Drive and caches Hugging Face models there across sessions
2. Installs dependencies
3. Clones (or pulls) the project code from GitHub
4. Builds the FAISS index from the finance knowledge base
5. Runs retrieval and safeguard sanity checks
6. Loads the generator and runs an end-to-end test
7. Launches the Gradio app (shareable public link)
8. Runs the evaluation suite and displays the results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME set to:", os.environ["HF_HOME"])

In [ ]:
!pip install sentence-transformers faiss-cpu transformers torch gradio accelerate --quiet

In [ ]:
import os
import sys

REPO_URL = "https://github.com/SheerinMM/finance-guidance-rag.git"
REPO_DIR = "/content/finance-guidance-rag"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Build the FAISS index

In [ ]:
from chunking import build_chunk_corpus
from embed_index import build_index, save_index

corpus = build_chunk_corpus("data/finance_kb.json")
print(f"Loaded {len(corpus)} chunks from knowledge base.")

index, embed_model, corpus = build_index(corpus)
save_index(index, corpus, "data/faiss_index.bin", "data/corpus.json")
print(f"Built FAISS index with {index.ntotal} vectors (dim={index.d}).")

## Retrieval sanity check

In [ ]:
from retrieve import retrieve

test_queries = [
    "What is a Lifetime ISA?",
    "How can I recognise a pension scam?",
    "What is a loan?",
    "How do I improve my credit score?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = retrieve(q, index, embed_model, corpus, top_k=3)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['title']} ({r['category']})")

## Safeguard test

In [ ]:
from safeguard import is_advice_request, check_safeguards

test_cases = [
    "Should I transfer my pension into a SIPP given my situation?",
    "What is a Lifetime ISA?",
    "What should I invest in given my situation?",
    "What is a loan?",
]

for q in test_cases:
    retrieved = retrieve(q, index, embed_model, corpus, top_k=3)
    blocked, message = check_safeguards(q, retrieved)
    print(f"Query: {q}")
    print(f"  is_advice_request: {is_advice_request(q)}")
    print(f"  Blocked: {blocked}")
    if blocked:
        print(f"  Message: {message}")
    print()

## Load the generation model

In [ ]:
from generate import Generator, format_answer_with_citations

generator = Generator()  # downloads Qwen2.5-1.5B-Instruct on first run (cached in HF_HOME)

## End-to-end test

In [ ]:
query = "What is a loan?"

retrieved = retrieve(query, index, embed_model, corpus, top_k=3)
blocked, message = check_safeguards(query, retrieved)

if blocked:
    print(message)
else:
    answer = generator.generate(query, retrieved)
    print(format_answer_with_citations(answer, retrieved))

## Launch the Gradio app

In [ ]:
import gradio as gr

from retrieve import retrieve
from safeguard import check_safeguards
from generate import Generator, format_answer_with_citations

TOP_K = 3
THEME = gr.themes.Soft(primary_hue="teal", secondary_hue="slate")

HEADER_HTML = """
<div style="background-color:#1a1a2e; padding:20px; border-radius:8px; text-align:center;">
  <h1 style="color:white; margin:0;">💷 Finance Guidance Assistant</h1>
</div>
"""

DISCLAIMER_HTML = """
<div style="background-color:#e6f4ea; border-left:4px solid #34a853; padding:12px 16px; border-radius:4px; margin-top:10px;">
  <strong>This is guidance, not regulated financial advice.</strong> Answers are grounded only in a
  small knowledge base of public guidance content. Personalised "what should I do" questions are
  redirected towards Pension Wise or a regulated financial adviser instead of being answered directly.
</div>
"""


def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, embed_model, corpus, top_k=TOP_K)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)


with gr.Blocks() as demo:
    gr.HTML(HEADER_HTML)
    gr.HTML(DISCLAIMER_HTML)
    gr.ChatInterface(
        fn=answer_question,
        editable=True,
        save_history=True,
        fill_height=True,
        examples=[
            "What is a Lifetime ISA?",
            "How can I recognise a pension scam?",
            "What is a loan?",
            "Should I transfer my whole pension into a SIPP?",  # triggers the safeguard
        ],
    )

demo.launch(theme=THEME, share=True)

## Run the evaluation suite

In [ ]:
!python eval/evaluate.py

## View evaluation results

These numbers go straight into the Part B technical report's evaluation section.

In [ ]:
import pandas as pd

retrieval_df = pd.read_csv("eval/retrieval_results.csv")
print("Retrieval accuracy:", retrieval_df["retrieval_hit"].mean())
print("Average retrieval latency (s):", retrieval_df["retrieval_latency_sec"].mean())
retrieval_df

In [ ]:
generation_df = pd.read_csv("eval/generation_results.csv")
generation_df
# Fill in the 'manual_grade' column (correct / partial / incorrect) by reading each
# generated_answer against the ground_truth, then re-load this CSV to compute
# an overall accuracy percentage for the report.

In [ ]:
safeguard_df = pd.read_csv("eval/safeguard_results.csv")
safeguard_df